<a href="https://colab.research.google.com/github/AhmedMahmoud-123/FlyRank_AI/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os

REPO_DIR = '/content/FlyRank_AI'

if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/AhmedMahmoud-123/FlyRank_AI.git

os.chdir(REPO_DIR)

!python scripts/01_prepare_features.py

import sys
sys.path.append('scripts')

from ml_utils import MODEL_NUMERIC_FEATURES

import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GroupShuffleSplit

df = pd.read_csv('data/processed/refresh_feature_vector.csv')
print(f'{len(df):,} rows loaded')

Prepared 30,000 rows from 30,000 raw rows
Wrote /content/FlyRank_AI/data/processed/refresh_feature_vector.csv
30,000 rows loaded


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

## 1. Paper findings

**Finding A — Growth Prediction (Part IV):** 90% accuracy on new pages from known brands,
75% on entirely unseen brands — a 15-point drop.

**Methodology questions:**
- What defines 'known brand' in the split, and could pages from the same brand appear in both train and test? (even different pages), and if so, does the model learn brand-level shortcuts
  (a brand's typical publishing cadence, niche, or baseline health score) rather than
  page-level growth signal? This is the same client-leakage question notebook 02 raised
  about `client_hash_id`.
- The report says this was "tested 20 different ways across both methods" with accuracy
  ranging 64%–85% on unseen brands — that's a wide range. What does the *distribution* look
  like, not just the average? A model that's 85% on some brand splits and 64% on others is
  a very different honesty story than a model that's consistently ~75%.
- Top predictor is "Days Visible" (0.16) — is that measured in a window that could overlap
  the label window (a page counted as "visible" during the same days used to judge whether
  it grew)? The paper doesn't show the feature-label timing explicitly.

**Finding B — Zombie Recovery (Part IV):** 99% same-brand vs 97% unseen-brand — only a
2-point gap, much tighter than Finding A's 15-point gap.

**Methodology questions:**
- Why does this model generalize to new brands so much better than Growth Prediction?
  One honest hypothesis: recovery may be driven by broadly transferable signals (content
  age, prior impressions) rather than brand-specific patterns — worth checking whether
  the top features here (Content Age, Impressions) are less brand-coupled than Growth
  Prediction's top feature (Days Visible).
- 59% of zero-traffic pages "came back on their own" — is "recovery" measured over a fixed
  window after the zero-traffic point, and is that window long enough that some "recoveries"
  are actually just noisy sparse-traffic pages crossing the zero threshold randomly rather
  than a real signal-driven comeback?
- The report is written by the company selling the automated fix for exactly this problem
  (zombie recovery). That's not a reason to dismiss the number, but it's a reason to check
  whether the reported metric (accuracy) is the most honest one — precision/recall on the
  minority "does NOT recover" class matters more for a triage tool than raw accuracy, and
  isn't reported here.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [2]:
feature_cols = [
    'impressions_prev_30d',
    'avg_position',
    'days_since_last_update'
]
model_data = df.dropna(subset=feature_cols)
X, y = model_data[feature_cols], model_data['is_declining_label']

# BEFORE: random split
Xr_tr, Xr_te, yr_tr, yr_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
rf_random = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(Xr_tr, yr_tr)
random_acc = rf_random.score(Xr_te, yr_te)

# AFTER: grouped split by client_id (Week 5's design)
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=model_data['client_id']))
Xg_tr, Xg_te = X.iloc[train_idx], X.iloc[test_idx]
yg_tr, yg_te = y.iloc[train_idx], y.iloc[test_idx]
rf_grouped = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(Xg_tr, yg_tr)
grouped_acc = rf_grouped.score(Xg_te, yg_te)

print(f'random split accuracy:  {random_acc:.3f}')
print(f'grouped split accuracy: {grouped_acc:.3f}')
print(f'base rate: {max(y.mean(), 1-y.mean()):.3f}')

random split accuracy:  0.656
grouped split accuracy: 0.594
base rate: 0.542


###Random split vs client-grouped split

The random split achieved 65.6% accuracy, while the client-grouped split achieved 59.4%, a 6.2 percentage-point drop when rows from the same client are kept on the same side of the split.

This indicates that a random row split gives a more optimistic result for this dataset. I therefore treat the client-grouped result as the more appropriate estimate for performance on clients not represented in training.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [4]:
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
import numpy as np

In [6]:
# Final W05 feature set
feature_cols = [
    'impressions_prev_30d',
    'avg_position',
    'days_since_last_update'
]

target_col = 'is_declining_label'

X = model_data[feature_cols].copy()
y = model_data[target_col].astype(int)
groups = model_data['client_id']

gkf = GroupKFold(n_splits=5)

rf = RandomForestClassifier(
    n_estimators=150,
    random_state=42,
    n_jobs=-1
)

score = cross_val_score(
    rf,
    X,
    y,
    groups=groups,
    cv=gkf,
    scoring='accuracy'
)

print("Grouped 5-fold accuracy by fold:")
print(np.round(score, 3))

print(f"Mean grouped CV accuracy: {score.mean():.3f}")
print(f"Std grouped CV accuracy:  {score.std():.3f}")
print(f"Base rate: {max(y.mean(), 1-y.mean()):.3f}")

Grouped 5-fold accuracy by fold:
[0.546 0.621 0.637 0.647 0.695]
Mean grouped CV accuracy: 0.629
Std grouped CV accuracy:  0.048
Base rate: 0.542


###Validation result

The final W05 three-feature model was evaluated using client-grouped 5-fold cross-validation. Accuracy across the five folds was 54.6%, 62.1%, 63.7%, 64.7%, and 69.5%, with a mean of 62.9% and a standard deviation of 4.8 percentage points.

The majority-class baseline is 54.2%, so the grouped validation result is approximately 8.7 percentage points above the baseline on average.

Performance varies across client folds, so the result should be treated as evidence of a useful directional signal rather than proof of strong generalization. The final feature set contains only the three features retained in W05 after the temporal-leakage review.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Original claim** The model achieves strong precision at identifying declining content

**Rewritten, matched to the evidence:**
On client-grouped 5-fold validation, the final three-feature model achieved 62.9% mean accuracy versus a 54.2% majority-class baseline, with fold accuracy ranging from 54.6% to 69.5%. This indicates a measurable but variable directional signal. The model should be used for decision-support and manual review prioritization, not treated as a guaranteed predictor of future decline.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.